# Final Competition-Grade GlyphMatics Tinker Submission — Gain 1.18

**Purpose:** Score-push variant after `0.85` result.

Changes from previous final notebook:

- Keeps `FORCED_FUSED_RANK = 32` for evaluator compatibility.
- Raises `SVD_ENERGY_GAIN_CAP` from `1.10` to `1.18`.
- Keeps offline/local Tinker conversion, local base model detection, marker creation, and clean `submission.zip`.

**Kaggle accelerator:** `GPU T4 x2`


## Required Kaggle Inputs

Attach these inputs before running:

1. Competition input: `NVIDIA Nemotron Model Reasoning Challenge`.
2. Base model: `nemotron-3-nano-30b-a3b-bf16`.
3. Adapter input: `huikang/nemotron-adapter`.
4. Tinker wheelhouse input containing `tinker`, `tinker-cookbook`, and `chz` wheels.

Internet is **not required** when these inputs are attached.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import json
import importlib.util

print("Python:", sys.version)
print("Working dir:", Path.cwd())

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

if not KAGGLE_INPUT.exists():
    raise RuntimeError("This notebook is intended to run inside Kaggle.")

KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)

def print_inputs():
    print("\n[Inputs]")
    for p in sorted(KAGGLE_INPUT.iterdir()):
        print(" -", p)

print_inputs()


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working dir: /kaggle/working

[Inputs]
 - /kaggle/input/competitions
 - /kaggle/input/datasets
 - /kaggle/input/models


## 1. Install or load Tinker locally

This scans every Kaggle input for local wheels and installs `tinker-cookbook` without using the Internet. If `tinker_cookbook` is already installed, it uses the existing installation.


In [2]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

def list_wheel_dirs():
    rows = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp")]:
        if not root.exists():
            continue
        for d in [root] + [p for p in root.rglob("*") if p.is_dir()]:
            wheels = sorted(d.glob("*.whl"))
            if wheels:
                rows.append((d, [w.name for w in wheels]))
    return rows

def score_tinker_dir(names):
    low = " ".join(n.lower() for n in names)
    score = 0
    for token in ["tinker_cookbook", "tinker-cookbook", "tinker_", "tinker-", "chz"]:
        if token in low:
            score += 1
    return score

def find_tinker_wheelhouse():
    candidates = []
    for d, names in list_wheel_dirs():
        score = score_tinker_dir(names)
        if score:
            candidates.append((score, len(names), d, names))
    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0] if candidates else None

print("[Tinker] scanning wheel folders...")
for d, names in list_wheel_dirs():
    interesting = [n for n in names if ("tinker" in n.lower() or "chz" in n.lower())]
    if interesting:
        print("\n[wheel-dir]", d)
        for name in interesting:
            print(" -", name)

if importlib.util.find_spec("tinker_cookbook") is not None:
    print("[Tinker] tinker_cookbook already installed; skipping wheel install")
else:
    explicit = os.environ.get("WHEEL_DIR")
    candidate = None

    if explicit and Path(explicit).exists():
        candidate = (999, 0, Path(explicit), [p.name for p in Path(explicit).glob("*.whl")])
    else:
        candidate = find_tinker_wheelhouse()

    if candidate is None:
        raise FileNotFoundError(
            "Could not find local tinker wheelhouse. Attach a Kaggle input containing "
            "tinker-cookbook/tinker/chz wheels or set WHEEL_DIR to that folder."
        )

    _, _, wheel_dir, names = candidate
    print("[Tinker] selected wheel_dir:", wheel_dir)

    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        f"--find-links={wheel_dir}",
        "tinker-cookbook",
        "tinker",
    ]
    print("[Tinker] pip:", " ".join(cmd))
    subprocess.run(cmd, check=True)

import importlib.metadata as md
import tinker_cookbook
from tinker_cookbook import weights

print("[Tinker] ready:", tinker_cookbook.__file__)
print("[Tinker] tinker_cookbook version:", md.version("tinker-cookbook"))
print("[Tinker] tinker version:", md.version("tinker"))
print("[Tinker] has build_lora_adapter:", hasattr(weights, "build_lora_adapter"))

if not hasattr(weights, "build_lora_adapter"):
    raise RuntimeError("tinker_cookbook.weights.build_lora_adapter not found")


[Tinker] scanning wheel folders...

[wheel-dir] /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
 - chz-0.4.0-py3-none-any.whl
 - tinker-0.18.1-py3-none-any.whl
 - tinker_cookbook-0.3.0-py3-none-any.whl
[Tinker] selected wheel_dir: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
[Tinker] pip: /usr/bin/python3 -m pip install --no-index --find-links=/kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse tinker-cookbook tinker
Looking in links: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker_cookbook-0.3.0-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker-0.18.1-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/chz-0.4.0-py3-none-any.whl (from tinker-cookbook)
[Tinker] ready: /usr/local/lib/python3.12/dist-packages/tinker_cookbook/__init__.py
[Tinker] tinker_cookbook version: 0.3.0
[T

## 2. Detect local model and adapter paths

This avoids `snapshot_download(...)` and keeps the run fully offline.


In [3]:
from pathlib import Path
import json

def find_first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return str(Path(p))
    return None

ADAPTER_PATH_CANDIDATES = [
    "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter",
]

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16",
]

ADAPTER_PATH = find_first_existing(ADAPTER_PATH_CANDIDATES)
BASE_MODEL_PATH = find_first_existing(BASE_MODEL_CANDIDATES)

if ADAPTER_PATH is None:
    # Generic fallback: adapter folder should contain adapter_config.json and adapter_model.safetensors.
    hits = []
    for cfg in Path("/kaggle/input").rglob("adapter_config.json"):
        folder = cfg.parent
        if (folder / "adapter_model.safetensors").exists():
            if "nemotron" in str(folder).lower() or "adapter" in str(folder).lower():
                hits.append(folder)
    if hits:
        ADAPTER_PATH = str(sorted(hits, key=lambda p: len(str(p)))[0])

if BASE_MODEL_PATH is None:
    # Generic fallback: local base model folder should contain config.json.
    hits = []
    for cfg in Path("/kaggle/input").rglob("config.json"):
        folder = cfg.parent
        s = str(folder).lower()
        if "nemotron" in s and ("30b" in s or "nano" in s):
            hits.append(folder)
    if hits:
        BASE_MODEL_PATH = str(sorted(hits, key=lambda p: len(str(p)))[0])

if ADAPTER_PATH is None:
    raise FileNotFoundError("Nemotron adapter input not found. Attach huikang/nemotron-adapter.")

if BASE_MODEL_PATH is None:
    raise FileNotFoundError("Local Nemotron base model not found. Attach nemotron-3-nano-30b-a3b-bf16.")

print("[Paths] ADAPTER_PATH:", ADAPTER_PATH)
print("[Paths] BASE_MODEL_PATH:", BASE_MODEL_PATH)

# Basic sanity checks.
if not (Path(ADAPTER_PATH) / "adapter_config.json").exists():
    raise FileNotFoundError(f"adapter_config.json missing under {ADAPTER_PATH}")

if not (Path(BASE_MODEL_PATH) / "config.json").exists():
    raise FileNotFoundError(f"config.json missing under {BASE_MODEL_PATH}")


[Paths] ADAPTER_PATH: /kaggle/input/models/huikang/nemotron-adapter/transformers/default/20
[Paths] BASE_MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


## 3. Apply GlyphMatics fused-projection transport patch

This patch preserves evaluator-compatible rank 32 output while applying conservative SVD energy restoration for fused projections. It also records a transport ledger for inspection.


In [4]:
from __future__ import annotations

from collections import Counter
import json
from pathlib import Path

import torch
import tinker_cookbook.weights._adapter as A

FORCED_FUSED_RANK = int(os.environ.get("FORCED_FUSED_RANK", "32"))
SVD_ENERGY_GAIN_CAP = float(os.environ.get("SVD_ENERGY_GAIN_CAP", "1.18"))

class GlyphmaticTransportLedger:
    def __init__(self):
        self.events = []
        self.counts = Counter()

    def emit(self, *, src, dst, op, alpha, beta=None, gamma=None):
        event = {
            "alpha": alpha,
            "source": str(src),
            "dest": str(dst),
            "op": str(op),
            "beta": beta or {},
            "gamma": gamma or {},
        }
        self.events.append(event)
        self.counts[(alpha, op)] += 1

    def markdown(self) -> str:
        lines = [
            "# GlyphMatics Transport Ledger",
            "",
            "Generated during tinker-cookbook adapter conversion.",
            "",
            "## Coordinate definition",
            "",
            "- **α**: module/transport family.",
            "- **β**: tensor geometry.",
            "- **γ**: preservation/compression action.",
            "",
            "## Event summary",
            "",
            "| α | op | count |",
            "|---|---:|---:|",
        ]
        for (alpha, op), count in sorted(self.counts.items()):
            lines.append(f"| `{alpha}` | `{op}` | {count} |")

        lines += [
            "",
            "## First 60 events",
            "",
            "| # | α | op | source | destination | γ |",
            "|---:|---|---|---|---|---|",
        ]
        for i, event in enumerate(self.events[:60], 1):
            gamma = json.dumps(event["gamma"], sort_keys=True)
            lines.append(
                f"| {i} | `{event['alpha']}` | `{event['op']}` | "
                f"`{event['source']}` | `{event['dest']}` | `{gamma}` |"
            )
        return "\n".join(lines) + "\n"

    def print_summary(self):
        print("[GlyphMatics ledger] events:", len(self.events))
        for (alpha, op), count in sorted(self.counts.items()):
            print(f"[GlyphMatics ledger] {alpha}:{op}={count}")


GLYPH_LEDGER = GlyphmaticTransportLedger()


def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int):
    """
    Compress Delta = B @ A to rank-k.
    Uses SVD plus conservative energy restoration to reduce under-scaling.
    """
    delta = B.float() @ A_mat.float()

    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
    total_mass = S.sum().clamp_min(1e-12)
    full_energy = torch.sqrt(torch.sum(S ** 2)).clamp_min(1e-12)

    U = U[:, :rank]
    S_k = S[:rank]
    Vh = Vh[:rank, :]

    kept_energy = torch.sqrt(torch.sum(S_k ** 2)).clamp_min(1e-12)
    gain = torch.clamp(full_energy / kept_energy, min=1.0, max=SVD_ENERGY_GAIN_CAP)

    sroot = torch.sqrt(S_k)
    B_new = (U * sroot.unsqueeze(0)) * gain
    A_new = sroot.unsqueeze(1) * Vh

    stats = {
        "rank_in": int(B.shape[1]),
        "rank_out": int(rank),
        "singular_mass_kept": float(S_k.sum() / total_mass),
        "energy_gain": float(gain),
    }

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous(), stats


def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    assert component_order is not None

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []
    merged_rank = 0
    row_offset = 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(
                f"Missing component {comp_name!r} for fused target {fused_model_key!r}"
            )

        lora_A, lora_B = comp_by_name[comp_name]
        r = lora_A.shape[0]
        out_dim = lora_B.shape[0]

        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r, comp_name))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(
        fused_out_dim,
        merged_rank,
        dtype=merged_lora_A.dtype,
        device=merged_lora_A.device,
    )

    rank_offset = 0
    for row_start, row_end, r, comp_name in comp_slices:
        _, lora_B = comp_by_name[comp_name]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    final_rank = merged_rank
    compression_stats = {
        "rank_in": int(merged_rank),
        "rank_out": int(merged_rank),
        "preservation": "exact",
    }

    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A, svd_stats = _compress_lora_pair_to_rank(
            merged_lora_B,
            merged_lora_A,
            FORCED_FUSED_RANK,
        )
        final_rank = FORCED_FUSED_RANK
        compression_stats = {
            **svd_stats,
            "preservation": "rank32_svd_energy_restored",
            "gain_cap": SVD_ENERGY_GAIN_CAP,
        }

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"

    GLYPH_LEDGER.emit(
        src=f"{adapter_layer_prefix}.{{{','.join(component_order)}}}",
        dst=peft_target_key,
        op="fused_projection_transport",
        alpha="mamba_or_fused_projection",
        beta={
            "fused_out_dim": int(fused_out_dim),
            "component_count": len(component_order),
            "component_order": list(component_order),
        },
        gamma=compression_stats,
    )

    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank


A._merge_fused_projections = patched_merge_fused_projections

print("[GlyphMatics] patched:", A._merge_fused_projections.__name__)
print("[GlyphMatics] FORCED_FUSED_RANK:", FORCED_FUSED_RANK)
print("[GlyphMatics] SVD_ENERGY_GAIN_CAP:", SVD_ENERGY_GAIN_CAP)


[GlyphMatics] patched: patched_merge_fused_projections
[GlyphMatics] FORCED_FUSED_RANK: 32
[GlyphMatics] SVD_ENERGY_GAIN_CAP: 1.18


## 4. Build adapter package

This clears stale output, builds the adapter using the local base model, writes the transport ledger, and guarantees the completion marker exists.


In [5]:
from pathlib import Path
import shutil
from tinker_cookbook import weights

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")

# Deterministic rerun: tinker refuses to write into an existing output folder.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

print("[Build] adapter_path:", ADAPTER_PATH)
print("[Build] base_model_path:", BASE_MODEL_PATH)
print("[Build] output_dir:", OUTPUT_DIR)

weights.build_lora_adapter(
    base_model=str(BASE_MODEL_PATH),
    adapter_path=str(ADAPTER_PATH),
    output_path=str(OUTPUT_DIR),
)

# Required marker. Some tinker-cookbook versions do not emit it.
if (OUTPUT_DIR / "adapter_config.json").exists() and (OUTPUT_DIR / "adapter_model.safetensors").exists():
    (OUTPUT_DIR / "checkpoint_complete").write_text("ok\n", encoding="utf-8")
else:
    raise FileNotFoundError("Adapter build did not produce adapter_config.json and adapter_model.safetensors")

# Review/debug metadata; harmless for human inspection.
GLYPH_LEDGER.print_summary()
ledger_text = GLYPH_LEDGER.markdown()
(OUTPUT_DIR / "GLYPHMATICS_TRANSPORT_LEDGER.md").write_text(ledger_text, encoding="utf-8")

readme = OUTPUT_DIR / "README.md"
if readme.exists():
    with open(readme, "a", encoding="utf-8") as f:
        f.write("\n\n")
        f.write(ledger_text)
else:
    readme.write_text(
        "# Nemotron Adapter Submission\n\n" + ledger_text,
        encoding="utf-8",
    )

print("[Build] output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name, p.stat().st_size)


[Build] adapter_path: /kaggle/input/models/huikang/nemotron-adapter/transformers/default/20
[Build] base_model_path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
[Build] output_dir: /kaggle/working/nemotron-adapter-ready-to-submit


MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


[GlyphMatics ledger] events: 23
[GlyphMatics ledger] mamba_or_fused_projection:fused_projection_transport=23
[Build] output files:
 - GLYPHMATICS_TRANSPORT_LEDGER.md 8154
 - README.md 8185
 - adapter_config.json 618
 - adapter_model.safetensors 3554384888
 - checkpoint_complete 3


## 5. Validate and create submission.zip

Only submit the resulting `/kaggle/working/submission.zip`.


In [6]:
import zipfile
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")
ZIP_PATH = Path("/kaggle/working/submission.zip")

required = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "README.md",
    "checkpoint_complete",
]

missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing required submission files: {missing}")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# Keep submission minimal but include the ledger as harmless documentation.
include = required + ["GLYPHMATICS_TRANSPORT_LEDGER.md"]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in include:
        p = OUTPUT_DIR / name
        if p.exists():
            zf.write(p, arcname=name)

print("[Zip] wrote:", ZIP_PATH)
print("[Zip] size:", ZIP_PATH.stat().st_size)
print("[Zip] contents:")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for name in zf.namelist():
        print(" -", name)

assert ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 0


[Zip] wrote: /kaggle/working/submission.zip
[Zip] size: 3270303046
[Zip] contents:
 - adapter_config.json
 - adapter_model.safetensors
 - README.md
 - checkpoint_complete
 - GLYPHMATICS_TRANSPORT_LEDGER.md


## 6. Final listing

After this cell, open the **Output** tab and submit `submission.zip`.


In [7]:
!ls -lah /kaggle/working/submission.zip /kaggle/working/nemotron-adapter-ready-to-submit


-rw-r--r-- 1 root root 3.1G May  3 07:41 /kaggle/working/submission.zip

/kaggle/working/nemotron-adapter-ready-to-submit:
total 3.4G
drwxr-xr-x 2 root root 4.0K May  3 07:38 .
drwxr-xr-x 3 root root 4.0K May  3 07:38 ..
-rw-r--r-- 1 root root  618 May  3 07:38 adapter_config.json
-rw-r--r-- 1 root root 3.4G May  3 07:38 adapter_model.safetensors
-rw-r--r-- 1 root root    3 May  3 07:38 checkpoint_complete
-rw-r--r-- 1 root root 8.0K May  3 07:38 GLYPHMATICS_TRANSPORT_LEDGER.md
-rw-r--r-- 1 root root 8.0K May  3 07:38 README.md
